# 02 - DenseNet121

**Owner:** Caolan  |  **Plane:** sagittal  |  **Sweep task:** acl

Plain DenseNet121 backbone for the architecture sweep. `Huang et al., CVPR 2017`.
DenseNet121 + CBAM is only run later **if DenseNet121 wins** the sweep.

## How to use
1. Run the **Colab bootstrap** cell.
2. Implement `src/model_factory.py` (DenseNet branch).
3. Train on the sagittal ACL task and report AUC.

> **GPU equivalent:** `for-gpu/run_cnn.py --backbone densenet121` (Stage 1 — backbone screening). DenseNet121 is one of the two backbones carried forward (see report Stage 1).

In [1]:
# ============================================================
# Colab bootstrap - run this cell first.
# ============================================================
import sys
from pathlib import Path

# Mount Google Drive when running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# Locate the "codes" folder (the one containing src/ and notebooks/).
if IN_COLAB:
    # TODO: set this to where you uploaded the "codes" folder on Drive.
    # CODES_DIR = Path('/content/drive/MyDrive/AI-for-MIA/codes')
    CODES_DIR = Path('/content/drive/Shareddrives/MRNet_Group3/codes')
else:
    # Local run: this notebook lives in codes/notebooks/, so go up one level.
    CODES_DIR = Path.cwd().parent

assert (CODES_DIR / 'src').exists(), (
    f"Could not find src/ under {CODES_DIR}. "
    "Edit CODES_DIR above to point to your 'codes' folder."
)

# Make `import src...` work.
if str(CODES_DIR) not in sys.path:
    sys.path.insert(0, str(CODES_DIR))

# Auto-reload edited src/*.py without restarting the kernel.
# %load_ext autoreload
# %autoreload 2

# Shared config resolves the data folder (sibling of "codes").
from src import config
print('CODES_DIR:', config.CODES_DIR)
print('DATA_DIR :', config.DATA_DIR)
assert config.DATA_DIR.exists(), (
    f"Data folder not found at {config.DATA_DIR}. Put the MRNet 'data' folder "
    "next to 'codes', or set os.environ['MRNET_DATA_DIR'] before importing config."
)
print('Setup OK - data folder found.')


Mounted at /content/drive
CODES_DIR: /content/drive/Shareddrives/MRNet_Group3/codes
DATA_DIR : /content/drive/Shareddrives/MRNet_Group3/data
Setup OK - data folder found.


In [2]:
from src.model_factory import build_model
model = build_model(backbone="densenet121", use_cbam=False)
print(model)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 187MB/s]


MRNetModel(
  (backbone): DenseNet(
    (features): Sequential(
      (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): ReLU(inplace=True)
      (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (denselayer2): _DenseLayer(
          (norm1): BatchNorm2d(96, eps=1e-05, momentum

In [ ]:
import torch

from src import config
from src.data_pipeline import build_dataloaders, set_seed
from src.model_factory import build_model
from src.training_utils import run_training

set_seed(config.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader, val_loader = build_dataloaders(task="acl", plane=config.DEFAULT_PLANE)

model = build_model(backbone="densenet121", use_cbam=False).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5)

history = run_training(
    model, train_loader, val_loader, optimizer, scheduler, device,
    num_epochs=50,
    accumulation_steps=8,
    early_stopping_patience=10,
    checkpoint_dir=str(config.CHECKPOINTS_DIR / "densenet121_plain"),
    task_name="densenet121_acl",
)

best_val_auc = max(history["val_auc"])
print(f"Best validation AUC (DenseNet121, plain, ACL, sagittal): {best_val_auc:.4f}")


[densenet121_acl] Computing class weights from training data...
  Class weight | neg: 922, pos: 208, pos_weight: 4.4327
[densenet121_acl] epoch:   0 | train loss: 1.1103 | train auc: 0.5776 | val loss: 1.4345 | val auc: 0.8173 | val f1: 0.7048 | sens: 0.6852 | spec: 0.7879
--------------------------------------------------------------------------------
Checkpoint saved: /content/drive/Shareddrives/MRNet_Group3/codes/results/checkpoints/densenet121_plain/best_densenet121_acl.pth (val AUC: 0.8173)
[densenet121_acl] epoch:   1 | train loss: 0.9575 | train auc: 0.7603 | val loss: 1.6096 | val auc: 0.7761 | val f1: 0.6731 | sens: 0.6481 | spec: 0.7727
--------------------------------------------------------------------------------
[densenet121_acl] epoch:   2 | train loss: 0.7518 | train auc: 0.8662 | val loss: 1.1546 | val auc: 0.8269 | val f1: 0.6964 | sens: 0.7222 | spec: 0.7121
--------------------------------------------------------------------------------
Checkpoint saved: /content/d

In [ ]:
# DenseNet121 scaffold. Owner: Caolan.
#
# from src.data_pipeline import build_dataloaders
# from src.model_factory import build_model
# from src.training_utils import set_seed, fit
# from src.evaluation import evaluate_model
#
# TODO (sweep entry - plain backbone):
#   set_seed(config.SEED)
#   train_loader, val_loader, test_loader = build_dataloaders(config.DEFAULT_PLANE, 'acl')
#   model = build_model(backbone='densenet121', use_cbam=False)
#   fit(model, train_loader, val_loader, ...)
#   evaluate_model(model, test_loader, device)
#
# TODO (only if DenseNet121 wins the sweep): rerun with use_cbam=True as the ablation.
